# Bilateral filtering

The bilateral filter smooths a raster while preserving edges. Unlike a simple mean filter, it weights each neighbor by both spatial distance and value similarity. Pixels across a sharp boundary contribute very little, so edges stay sharp while flat areas get smoothed.

Two parameters control the behavior:
- **sigma_spatial**: how far the spatial Gaussian reaches (kernel radius = ceil(2 * sigma_spatial))
- **sigma_range**: how much value difference is tolerated before a neighbor gets downweighted

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import bilateral
from xrspatial import mean
from xrspatial.terrain import generate_terrain

## Generate a synthetic terrain with noise

We'll create a DEM, add Gaussian noise, and then compare bilateral filtering against the standard mean filter.

In [ ]:
W, H = 600, 400
cvs_terrain = xr.DataArray(
    np.zeros((H, W)),
    dims=['y', 'x'],
    coords={'y': np.linspace(0, 100, H), 'x': np.linspace(0, 150, W)},
)
terrain = generate_terrain(cvs_terrain, seed=42)

# Add noise
rng = np.random.default_rng(123)
noise = rng.normal(0, 15, terrain.shape)
noisy_terrain = terrain.copy(data=terrain.values + noise)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
terrain.plot(ax=axes[0], cmap='terrain')
axes[0].set_title('Clean terrain')
noisy_terrain.plot(ax=axes[1], cmap='terrain')
axes[1].set_title('Noisy terrain')
plt.tight_layout()

## Compare bilateral vs. mean filter

The mean filter blurs edges. The bilateral filter preserves them.

In [ ]:
smoothed_bilateral = bilateral(noisy_terrain, sigma_spatial=2.0, sigma_range=20.0)
smoothed_mean = mean(noisy_terrain, passes=3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

noisy_terrain.plot(ax=axes[0], cmap='terrain')
axes[0].set_title('Noisy input')

smoothed_mean.plot(ax=axes[1], cmap='terrain')
axes[1].set_title('Mean filter (3 passes)')

smoothed_bilateral.plot(ax=axes[2], cmap='terrain')
axes[2].set_title('Bilateral filter')

plt.tight_layout()

## Effect of sigma_range

Smaller `sigma_range` preserves more edges; larger values allow smoothing across bigger value differences.

In [ ]:
sigma_ranges = [5.0, 20.0, 100.0]

fig, axes = plt.subplots(1, len(sigma_ranges), figsize=(18, 5))
for ax, sr in zip(axes, sigma_ranges):
    result = bilateral(noisy_terrain, sigma_spatial=2.0, sigma_range=sr)
    result.plot(ax=ax, cmap='terrain')
    ax.set_title(f'sigma_range = {sr}')
plt.tight_layout()

## Step-edge preservation

A clear demonstration: a raster with a sharp vertical edge. The bilateral filter keeps the boundary; the mean filter blurs it.

In [ ]:
step = np.zeros((50, 100))
step[:, 50:] = 100.0

# Add a bit of noise
step_noisy = step + rng.normal(0, 5, step.shape)
step_agg = xr.DataArray(step_noisy, dims=['y', 'x'])

step_bilateral = bilateral(step_agg, sigma_spatial=2.0, sigma_range=10.0)
step_mean = mean(step_agg, passes=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
step_agg.plot(ax=axes[0], cmap='gray')
axes[0].set_title('Noisy step edge')
step_mean.plot(ax=axes[1], cmap='gray')
axes[1].set_title('Mean filter')
step_bilateral.plot(ax=axes[2], cmap='gray')
axes[2].set_title('Bilateral filter')
plt.tight_layout()

# Cross-section
row = 25
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(step_agg.data[row], label='Noisy', alpha=0.5)
ax.plot(step_mean.data[row], label='Mean', linewidth=2)
ax.plot(step_bilateral.data[row], label='Bilateral', linewidth=2)
ax.legend()
ax.set_xlabel('Column')
ax.set_ylabel('Value')
ax.set_title('Cross-section at row 25')
plt.tight_layout()